<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 1: From a Case Description to a Formulation

**A formulation makes the choice, response, comparison rule, and requirements precise enough to evaluate any candidate.**

Lecture 01-2 identified decision variables, objectives, and constraints in five everyday cases. This lecture keeps those cases and gives each one standard mathematical notation. Part 1 introduces the common template through Case A, the digital clock.

### 1 · Use one standard template

Let \(x\) be the decision vector and let \(\mathcal X\) be its allowed domain. A system model or direct calculation produces responses \(y=\operatorname{Sim}(x)\). A scalar objective \(f\) compares feasible candidates. Inequality and equality residuals express requirements.

> $\displaystyle \underset{x\in\mathcal X}{\operatorname{minimize}}\quad f(y)$
>
> $\displaystyle \text{subject to}\quad y=\operatorname{Sim}(x),$
>
> $\displaystyle g_j(x,y)\le0\quad(j=1,\ldots,m_g),$
>
> $\displaystyle h_k(x,y)=0\quad(k=1,\ldots,m_h).$

Here, \(m_g\) and \(m_h\) are the numbers of inequality and equality constraints. Maximization can be written in the minimization template by negating the quantity to be maximized.

For a dynamic system, \(\operatorname{Sim}\) can repeat the physical state transition \(F\) and then apply the performance mapping \(G\). For the classroom, the standard-form objective \(f(y;\lambda_E)\) is the score supplied by \(H\): \(f(y;\lambda_E)=J(u;\lambda_E)\). Uppercase \(F\) remains reserved for the physical state transition.

### 2 · Formulate Case A: correct the clock

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_a_clock.png" alt="A classroom clock reading 09:59, a reference clock reading 10:00, and an adjustable time-correction control." width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Case A from 01-2: the clock correction becomes the decision \(x=[q]\). The remaining errors are the responses \(y\).

The observed clock errors are \(d=(-2,-1,-1,0)\) minutes. The stored correction \(q\) is the real choice. In standard notation,

| Formulation part | Clock case |
|:---|:---|
| Decision | \(x=[q]\) |
| Domain | \(\mathcal X=\mathbb R\) |
| Response | \(y=\operatorname{Sim}(x)=d+q\mathbf 1\) |
| Objective | \(f(y)=\sum_{i=1}^{4}y_i^2\) |
| Constraints | None, so \(m_g=m_h=0\) |

The complete formulation is

> $\displaystyle \underset{q\in\mathbb R}{\operatorname{minimize}}\quad
\sum_{i=1}^{4}(d_i+q)^2.$

The vector \(y\) contains the remaining errors. It is produced by the chosen correction and is not another decision.

The next cell defines the fixed observations, response calculation, and objective separately.

In [ ]:
import numpy as np

# Fixed observations
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])


def simulate_clock(x):
    '''Return y=Sim(x): remaining clock errors after the correction.'''
    correction = float(np.asarray(x)[0])
    return OBSERVED_ERRORS + correction


def objective_function(y):
    '''Return the sum of squared remaining errors.'''
    return float(np.sum(np.asarray(y) ** 2))


def evaluate_candidate(x):
    '''Evaluate one unconstrained clock-correction candidate.'''
    decision = np.asarray(x, dtype=float)
    response = simulate_clock(decision)
    return {"x": decision, "y": response, "f": objective_function(response)}

The left panel shows the four remaining errors. The right panel places the same correction on the squared-error objective curve. The orange marker is the current correction; the gold star is the best candidate on the stated 0.25-minute search grid.

At \(q=0\), the errors are unchanged and \(f=6\). At \(q=1\), the errors become \((-1,0,0,1)\) and \(f=2\). A single correction changes every error by the same amount.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_a_formulation_graph.png" alt="Linked error bars and a squared-error curve show how a clock correction shifts all four errors and moves the current objective value." width="1000" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
import sys
import matplotlib


def _pyplot(*, interactive=False):
    """Use the course's widget-backend fallback outside the browser runtime."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
PURPLE = "#8856a7"
GRAY = "#a6a6a6"
GOLD = "#f6c945"


def case_canvas(title, controls, *, panels=2, interactive=False):
    """Create a shared layout; controls are (name, label, low, high, value, step, color)."""
    plt = _pyplot(interactive=interactive)
    figure, axes = plt.subplots(
        1, panels, figsize=(12.4 if panels == 2 else 14.0, 7.4 if interactive else 5.3)
    )
    figure.suptitle(title, y=0.985, fontsize=14, fontweight="bold")
    status = figure.text(0.5, 0.918, "", ha="center", va="center", fontsize=11)
    footer = figure.text(0.5, 0.045 if not interactive else 0.27, "",
                         ha="center", va="center", fontsize=9)
    sliders = {}
    if interactive:
        from matplotlib.widgets import Slider

        figure.subplots_adjust(left=0.075, right=0.97,
                               bottom=0.41 if panels == 3 else 0.37, top=0.83, wspace=0.38)
        positions = np.linspace(0.19, 0.065, max(len(controls), 2))
        for position, (name, label, low, high, value, step, color) in zip(positions, controls):
            slider_axis = figure.add_axes([0.28, position, 0.61, 0.026])
            sliders[name] = Slider(slider_axis, label, low, high, valinit=value,
                                   valstep=step, valfmt="%1.0f" if step >= 1 else "%1.2f",
                                   color=color, initcolor=color)
    figure._case_sliders = sliders
    figure._case_state = {}
    return plt, figure, axes, sliders, status, footer


def finish_case(plt, figure, axes, sliders, refresh, *, interactive=False):
    """Connect controls and keep widgets and evaluated records alive on the figure."""
    for axis in axes:
        axis.grid(alpha=0.25)
    for slider in sliders.values():
        slider.on_changed(refresh)
    figure._case_refresh = refresh
    refresh()
    if not interactive:
        figure.tight_layout(rect=(0.015, 0.09, 0.985, 0.96))
    plt.show()
    if not interactive:
        plt.close(figure)
    return figure


def show_clock_formulation(correction=0.0, *, interactive=False):
    controls = [("correction", "Decision: correction q (min)", -1.0, 2.0, correction, 0.05, BLUE)]
    plt, figure, axes, sliders, status, footer = case_canvas(
        "Case A · One correction changes four errors and one score",
        controls, interactive=interactive,
    )
    observations = np.arange(1, 5)
    axes[0].bar(observations - 0.18, OBSERVED_ERRORS, width=0.36,
                color="#c5d8e8", label="Before correction")
    error_bars = axes[0].bar(observations + 0.18, OBSERVED_ERRORS, width=0.36,
                             color=ORANGE, label="After correction")
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set(xlabel="Comparison number", ylabel="Remaining error (min)",
                xticks=observations, ylim=(-3.4, 2.7), title="Each error shifts by the same q")
    axes[0].legend(fontsize=9, loc="upper left")
    curve_grid = np.linspace(-1.0, 2.0, 201)
    curve_scores = [evaluate_candidate([value])["f"] for value in curve_grid]
    search_grid = np.arange(-1.0, 2.0 + 0.125, 0.25)
    records = [evaluate_candidate([value]) for value in search_grid]
    best = min(records, key=lambda record: record["f"])
    axes[1].plot(curve_grid, curve_scores, color=BLUE, linewidth=2, label="Squared-error score")
    axes[1].scatter(search_grid, [r["f"] for r in records], color=TEAL, s=24,
                    label="0.25-min search grid")
    axes[1].scatter(best["x"][0], best["f"], marker="*", s=230, color=GOLD,
                    edgecolor="black", zorder=5, label="Best grid candidate")
    current_marker = axes[1].scatter([], [], s=95, color=ORANGE, edgecolor="black",
                                     zorder=6, label="Current correction")
    current_line = axes[1].axvline(correction, color=ORANGE, linestyle=":", linewidth=1.5)
    axes[1].set(xlabel="Clock correction q (min)", ylabel="Squared-error score f (min²)",
                xlim=(-1.1, 2.1), ylim=(0, 20), title="The objective adds the squared errors")
    axes[1].legend(fontsize=8, loc="upper right")
    footer.set_text("The displayed range and slider steps are interface settings; the formulation allows q in R.")

    def refresh(_=None):
        value = sliders["correction"].val if sliders else correction
        current = evaluate_candidate([value])
        for bar, remaining_error in zip(error_bars, current["y"]):
            bar.set_height(remaining_error)
        current_marker.set_offsets([[value, current["f"]]])
        current_line.set_xdata([value, value])
        status.set_text(f"Current q = {value:.2f} min   |   f = {current['f']:.2f} min²"
                        f"   |   Best grid: q = {best['x'][0]:.2f}, f = {best['f']:.2f}")
        figure._case_state.update(current=current, best=best)
        figure.canvas.draw_idle()

    return finish_case(plt, figure, axes, sliders, refresh, interactive=interactive)

In [ ]:
static_figure = show_clock_formulation(correction=0.0)

The blue trackbar changes \(q\). The orange error bars and objective marker update together. The displayed interval \([-1,2]\) and 0.05-minute slider step are interface settings; the optimization domain remains \(\mathbb R\).

The gold star is a search-grid benchmark. For these observations, \(f=4(q-1)^2+2\), so the continuous minimum is also at \(q=1\).

In [ ]:
formulation_explorer = show_clock_formulation(
    correction=0.0, interactive=True
)

### 3 · Classify only after the formulation is complete

| Classification axis | Clock formulation | Evidence |
|:---|:---|:---|
| Constraints | Unconstrained | No explicit bounds, inequalities, or equalities |
| Decision domain | Continuous | \(q\in\mathbb R\) |
| Objectives | Single-objective | One scalar sum is minimized |
| Function structure | Nonlinear | The objective contains squared terms |
| Response evaluation | Direct algebraic | The remaining errors are calculated directly |
| Uncertainty | Deterministic | One fixed set of observed errors is used |

The reported grid result is the best candidate on that finite grid. It is not, by itself, proof of the continuous optimum. Here the expression \(f=4(q-1)^2+2\) also establishes the continuous minimizer.

If the domain changes to \([-1,1]\), the stored correction still changes the real clock. The bound changes only which corrections are eligible.

### Takeaway

Translate a case in a fixed order:

> **decision \(x\) and domain \(\mathcal X\) → response \(y=\operatorname{Sim}(x)\) → feasibility from \(g,h\) → comparison by \(f\) → problem classification**

Case A has \(m_g=m_h=0\), so feasibility is automatic for every \(q\in\mathbb R\). Part 2 applies the same template to a case with spending bounds and a coupled budget requirement.